In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
import scipy.sparse.linalg as spla

# Notes

In [ ]:
def fdm_neg_lapl_unif_disc_mat(interior_shape, non_dirichlet_bd=None):
    num_dims = len(interior_shape)
    if non_dirichlet_bd is None:
        non_dirichlet_bd = num_dims * ((False, False),)

    assert len(non_dirichlet_bd) == num_dims, f"mismatch in dimensions"

    neg_lapl = sparse.dia_array((0, 0))
    for i, (dim_size, (is_low_bd_nd, is_high_bd_nd)) in enumerate(
        zip(reversed(interior_shape), reversed(non_dirichlet_bd))
    ):
        if i == 0:
            neg_lapl = sparse.diags_array(
                [2 * num_dims, -1, -1], offsets=[0, 1, -1], shape=(dim_size, dim_size)
            )
        else:
            neg_lapl = sparse.block_array(
                [
                    [
                        (
                            neg_lapl
                            if i == j
                            else (
                                -sparse.eye_array(*neg_lapl.shape)
                                if abs(i - j) == 1
                                else None
                            )
                        )
                        for j in range(dim_size)
                    ]
                    for i in range(dim_size)
                ]
            )
    return neg_lapl


fig, ax = plt.subplots()
ax: plt.Axes = ax

ax_im = ax.matshow(fdm_neg_lapl_unif_disc_mat((5, 4, 3)).toarray())
fig.colorbar(ax_im)